In [1]:
import pandas as pd
import numpy as np

In [2]:
cc_calls = pd.read_csv(
    "../../data/01_raw/raw_cc_calls.csv",
    low_memory=False
)

print(f"Shape: {cc_calls.shape}")
cc_calls.shape

Shape: (32882, 33)


(32882, 33)

In [3]:
cc_calls.info()
cc_calls.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32882 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Contact_ID                                32882 non-null  float64
 1   Call_Date                                 32882 non-null  object 
 2   Direction                                 32882 non-null  object 
 3   cc_care_package                           32744 non-null  object 
 4   cc_care_package_discussed                 32744 non-null  object 
 5   cc_urgency_getting_on_site                32744 non-null  object 
 6   cc_external_consultant                    32744 non-null  object 
 7   cc_agent_cross_sell_attempt               32744 non-null  object 
 8   cc_customer_issues_concerns               32744 non-null  object 
 9   cc_business_struggles_financial_hardship  32744 non-null  object 
 10  cc_call_initiated_by              

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,08-05-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,25-11-2024,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,23-10-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,13-01-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,19-03-2025,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025


In [4]:
cc_calls = cc_calls.drop_duplicates()
print("Shape after duplicate removal:", cc_calls.shape)

Shape after duplicate removal: (32789, 33)


In [5]:
cc_calls.columns = (
    cc_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [6]:
obj_cols = cc_calls.select_dtypes(include="object").columns

for col in obj_cols:
    cc_calls[col] = cc_calls[col].astype(str).str.strip()

In [7]:
date_cols = [
    "call_date",
    "created_date",
    "updated_date"
]

for col in date_cols:
    if col in cc_calls.columns:
        cc_calls[col] = pd.to_datetime(
            cc_calls[col],
            errors="coerce"
        )

In [8]:
possible_numeric_cols = [
    "call_duration",
    "number_of_attempts",
    "score",
    "customer_rating"
]

for col in possible_numeric_cols:
    if col in cc_calls.columns:
        cc_calls[col] = pd.to_numeric(
            cc_calls[col],
            errors="coerce"
        )

In [9]:
cat_cols = cc_calls.select_dtypes(include="object").columns

for col in cat_cols:
    cc_calls[col] = cc_calls[col].replace("nan", np.nan)
    cc_calls[col] = cc_calls[col].fillna("Unknown")

In [10]:
num_cols = cc_calls.select_dtypes(include=np.number).columns

for col in num_cols:
    cc_calls[col] = cc_calls[col].fillna(
        cc_calls[col].median()
    )

In [11]:
cc_calls.isnull().sum().sort_values(ascending=False).head(20)

call_date                                19194
cc_contractor_sentiment_issues_score         0
cc_questions_harder_than_expected            0
cc_dissatisfaction_support                   0
cc_contractor_sentiment                      0
cc_contractor_sentiment_start_score          0
cc_contractor_sentiment_end_score            0
cc_contractor_sentiment_overall_score        0
contact_id                                   0
cc_process_complexity_concerns               0
cc_pricing_sentiment_impact                  0
cc_refund_discussed                          0
cc_contractor_suggest_leave                  0
cc_contractor_complained                     0
co_ref                                       0
analysed_call                                0
cc_pricing_mentioned                         0
cc_dissatisfaction_time_to_complete          0
cc_platform_issues                           0
cc_login_issues                              0
dtype: int64

In [12]:
# 🔍 COMPREHENSIVE CC_CALLS DATA QUALITY ANALYSIS

print("="*70)
print("CC_CALLS DATA: QUALITY & COLUMN ANALYSIS")
print("="*70)

print(f"\nDataset Shape: {cc_calls.shape}")
print(f"Total nulls: {cc_calls.isnull().sum().sum()}\n")

# 1. Check ID/Tracking columns
print("1️⃣  ID/TRACKING COLUMNS (cardinality check):")
id_keywords = ["id", "ref", "call_id", "unnamed", "row"]
for col in cc_calls.columns:
    if any(keyword in col.lower() for keyword in id_keywords):
        unique_pct = cc_calls[col].nunique() / len(cc_calls) * 100
        print(f"   {col}: {cc_calls[col].nunique()} unique ({unique_pct:.1f}%)")

# 2. Check for completely empty/sparse columns
print("\n2️⃣  EMPTY/SPARSE COLUMNS (>80% missing):")
sparse_found = False
for col in cc_calls.columns:
    missing_pct = cc_calls[col].isnull().sum() / len(cc_calls) * 100
    if missing_pct > 80:
        print(f"   {col}: {missing_pct:.1f}% missing")
        sparse_found = True
if not sparse_found:
    print("   ✅ None found")

# 3. Check low-variance columns (>90% one value)
print("\n3️⃣  LOW-VARIANCE COLUMNS (>90% single value):")
low_var_found = False
for col in cc_calls.select_dtypes(include="object").columns:
    if cc_calls[col].nunique() > 0:
        top_pct = cc_calls[col].value_counts().head(1).values[0] / len(cc_calls) * 100
        if top_pct > 90:
            top_val = cc_calls[col].value_counts().head(1).index[0]
            print(f"   {col}: '{top_val}' = {top_pct:.1f}%")
            low_var_found = True
if not low_var_found:
    print("   ✅ None found")

# 4. Check "Unknown" concentration
print("\n4️⃣  'UNKNOWN' VALUE CONCENTRATION:")
unknown_cols = []
for col in cc_calls.select_dtypes(include="object").columns:
    unknown_pct = (cc_calls[col] == "Unknown").sum() / len(cc_calls) * 100
    if unknown_pct > 30:
        print(f"   {col}: {unknown_pct:.1f}% Unknown")
        unknown_cols.append(col)

if not unknown_cols:
    print("   ✅ No excessive Unknown values found")

# 5. Check numeric columns
print("\n5️⃣  NUMERIC COLUMNS - Value Distribution:")
for col in cc_calls.select_dtypes(include=[np.number]).columns:
    zero_count = (cc_calls[col] == 0).sum()
    zero_pct = zero_count / len(cc_calls) * 100
    print(f"   {col}: min={cc_calls[col].min():.2f}, max={cc_calls[col].max():.2f}, zeros={zero_pct:.1f}%")

# 6. Column type summary
print("\n6️⃣  COLUMN TYPE DISTRIBUTION:")
print(f"   Categorical (object): {cc_calls.select_dtypes(include='object').shape[1]}")
print(f"   Numeric (int/float): {cc_calls.select_dtypes(include=[np.number]).shape[1]}")
print(f"   Datetime: {cc_calls.select_dtypes(include=['datetime64']).shape[1]}")

print("\n" + "="*70)

CC_CALLS DATA: QUALITY & COLUMN ANALYSIS

Dataset Shape: (32789, 33)
Total nulls: 19194

1️⃣  ID/TRACKING COLUMNS (cardinality check):
   contact_id: 496 unique (1.5%)
   cc_refund_discussed: 6 unique (0.0%)
   co_ref: 14989 unique (45.7%)

2️⃣  EMPTY/SPARSE COLUMNS (>80% missing):
   ✅ None found

3️⃣  LOW-VARIANCE COLUMNS (>90% single value):
   cc_agent_cross_sell_attempt: 'No' = 95.8%
   cc_business_struggles_financial_hardship: 'No' = 96.5%
   cc_login_issues: 'No' = 94.6%
   cc_platform_issues: 'No' = 92.8%
   cc_dissatisfaction_time_to_complete: 'No' = 95.4%
   cc_questions_harder_than_expected: 'No' = 99.5%
   cc_dissatisfaction_support: 'No' = 97.6%
   cc_pricing_sentiment_impact: 'No' = 96.4%
   cc_refund_discussed: 'No' = 99.1%
   cc_contractor_suggest_leave: 'No' = 97.0%
   cc_contractor_complained: 'No' = 92.3%

4️⃣  'UNKNOWN' VALUE CONCENTRATION:
   ✅ No excessive Unknown values found

5️⃣  NUMERIC COLUMNS - Value Distribution:
   contact_id: min=193615000000.00, max=6918

In [13]:

# 🧹 ADDITIONAL CC_CALLS CLEANING

print("\n📋 PERFORMING ADDITIONAL CLEANING...\n")

# 1. Drop ID columns with very high cardinality
cols_to_drop = []
for col in cc_calls.columns:
    if col.lower() in ["co_ref", "customer_ref", "call_id"]:
        unique_pct = cc_calls[col].nunique() / len(cc_calls) * 100
        if unique_pct > 30:
            cols_to_drop.append(col)
            print(f"   ✅ Dropping '{col}' (high cardinality: {unique_pct:.1f}%)")

if cols_to_drop:
    cc_calls = cc_calls.drop(columns=cols_to_drop, errors='ignore')

# 2. Drop extremely low-variance columns (>95% Unknown)
low_var_cols = []
for col in cc_calls.select_dtypes(include="object").columns:
    if cc_calls[col].nunique() > 0:
        top_pct = cc_calls[col].value_counts().head(1).values[0] / len(cc_calls) * 100
        if top_pct > 95 and cc_calls[col].value_counts().head(1).index[0] == "Unknown":
            low_var_cols.append(col)
            print(f"   ✅ Dropping '{col}' ({top_pct:.1f}% Unknown)")

if low_var_cols:
    cc_calls = cc_calls.drop(columns=low_var_cols)

# 3. Convert binary Yes/No columns to numeric
print(f"\n   Converting binary columns to numeric...")
binary_count = 0
for col in cc_calls.select_dtypes(include="object").columns:
    unique_vals = set(cc_calls[col].dropna().unique())
    if unique_vals.issubset({'Yes', 'No'}) and len(unique_vals) > 1:
        cc_calls[col] = cc_calls[col].map({'Yes': 1, 'No': 0}).astype('int64')
        binary_count += 1

if binary_count > 0:
    print(f"   ✅ Converted {binary_count} binary columns")

# 4. Final null check
null_check = cc_calls.isnull().sum().sum()
print(f"\n   Final nulls: {null_check}")

# 5. Summary
print(f"\n✅ AFTER ADDITIONAL CLEANING:")
print(f"   Shape: {cc_calls.shape}")
print(f"   Columns removed: {len(cols_to_drop) + len(low_var_cols)}")
print(f"   Binary columns converted: {binary_count}")

# 6. Save cleaned data
cc_calls.to_csv(
    "../../data/02_processed/processed_cc_calls.csv",
    index=False
)
print(f"\n   ✅ Saved to processed_cc_calls.csv")


📋 PERFORMING ADDITIONAL CLEANING...

   ✅ Dropping 'co_ref' (high cardinality: 45.7%)

   Converting binary columns to numeric...

   Final nulls: 19194

✅ AFTER ADDITIONAL CLEANING:
   Shape: (32789, 32)
   Columns removed: 1
   Binary columns converted: 0

   ✅ Saved to processed_cc_calls.csv


In [14]:

# 📊 FINAL VERIFICATION

print("\n📊 FINAL CC_CALLS DATA STATUS:\n")

# Reload the saved data
final_cc_calls = pd.read_csv("../../data/02_processed/processed_cc_calls.csv")

print(f"Shape: {final_cc_calls.shape}")
print(f"Nulls: {final_cc_calls.isnull().sum().sum()}")
print(f"Memory: {final_cc_calls.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nColumn types:")
col_types = final_cc_calls.dtypes.value_counts()
for dtype, count in col_types.items():
    print(f"   {dtype}: {count}")

print(f"\n✅ CC_CALLS data is clean and ready for modeling!")


📊 FINAL CC_CALLS DATA STATUS:

Shape: (32789, 32)
Nulls: 19194
Memory: 48.04 MB

Column types:
   object: 29
   int64: 2
   float64: 1

✅ CC_CALLS data is clean and ready for modeling!


In [15]:

# 🔄 FINAL NULL HANDLING

print("\n🔄 HANDLING REMAINING NULLS...\n")

cc_calls_processed = pd.read_csv("../../data/02_processed/processed_cc_calls.csv")

print(f"Before final null handling: {cc_calls_processed.isnull().sum().sum()} nulls")

# Handle datetime columns with NaT
for col in cc_calls_processed.columns:
    if cc_calls_processed[col].dtype == 'object':
        # Try to detect if this looks like a date column
        sample = cc_calls_processed[col].dropna().head(5)
        if len(sample) > 0:
            try:
                # If it can be converted to datetime, fill NaT values
                test_convert = pd.to_datetime(sample, errors='coerce')
                if test_convert.notna().all():
                    cc_calls_processed[col] = pd.to_datetime(cc_calls_processed[col], errors='coerce')
                    nat_count = cc_calls_processed[col].isna().sum()
                    if nat_count > 0:
                        # Fill with a default date
                        cc_calls_processed[col] = cc_calls_processed[col].fillna(cc_calls_processed[col].median())
                        print(f"   ✅ Filled {nat_count} NaT values in '{col}'")
            except:
                pass

# Fill remaining numeric nulls
for col in cc_calls_processed.select_dtypes(include=[np.number]).columns:
    if cc_calls_processed[col].isnull().sum() > 0:
        cc_calls_processed[col] = cc_calls_processed[col].fillna(cc_calls_processed[col].median())
        print(f"   ✅ Filled numeric nulls in '{col}'")

print(f"\nAfter final null handling: {cc_calls_processed.isnull().sum().sum()} nulls")

# Save final cleaned data
cc_calls_processed.to_csv(
    "../../data/02_processed/processed_cc_calls.csv",
    index=False
)

print(f"\n✅ FINAL CC_CALLS STATUS:")
print(f"   Shape: {cc_calls_processed.shape}")
print(f"   Nulls: {cc_calls_processed.isnull().sum().sum()}")
print(f"   Memory: {cc_calls_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\n   ✅ Saved to processed_cc_calls.csv")


🔄 HANDLING REMAINING NULLS...

Before final null handling: 19194 nulls
   ✅ Filled 19194 NaT values in 'call_date'


C:\Users\shrey\AppData\Local\Temp\ipykernel_4592\2941152472.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_convert = pd.to_datetime(sample, errors='coerce')
C:\Users\shrey\AppData\Local\Temp\ipykernel_4592\2941152472.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_convert = pd.to_datetime(sample, errors='coerce')
C:\Users\shrey\AppData\Local\Temp\ipykernel_4592\2941152472.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_convert = pd.to_datetime(sample, errors='coerce')
C:\Users\shrey\AppData\Local\Temp\ipykernel_4592\2941152472.py:17: UserWarni


After final null handling: 0 nulls

✅ FINAL CC_CALLS STATUS:
   Shape: (32789, 32)
   Nulls: 0
   Memory: 46.94 MB

   ✅ Saved to processed_cc_calls.csv
